## Special Applications: Face recognition $ Neural Style Transfer

Explore how CNNs can be applied to multiple fields, including art generation and face recognition, then implement your own algorithm to generate art and recognize faces!

**Learning Objectives**
* Differentiate between face recognition and face verification
* Implement one-shot learning to solve a face recognition problem
* Apply the triplet loss function to learn a network's parameters in the context of face recognition
* Explain how to pose face recognition as a binary classification problem
* Map face images into 128-dimensional encodings using a pretrained model
* Perform face verification and face recognition with these encodings
* Implement the Neural Style Transfer algorithm
* Generate novel artistic images using Neural Style Transfer
* Define the style cost function for Neural Style Transfer
* Define the content cost function for Neural Style Transfer

---

## Table of Contents

---

## What is Face Recognition?

This section marks the transition into the final week of the course, moving from general architecture to specialized Computer Vision applications like **Face Recognition** and **Neural Style Transfer**.

### The "Multiplication of Error" Challenge

Recognition is mathematically much harder than Verification:

* If a Verification system has a $99\%$ accuracy ($1\%$ error rate), it sounds good for a single check.
* However, if we run that same system against a database of $K = 100$ people for a Recognition task, we have $100$ chances to make a "False Positive" mistake.
* To have a reliable Recognition system for a large database, our underlying Verification building block must achieve extremely high accuracy (e.g., $99.9\%+$).

### Preview of the "One-Shot Learning" Problem

The core difficulty in face recognition for most organizations is that we often have only one image of an employee or user to train on. Standard CNNs usually require thousands of images to learn a class.

In the upcoming sections, we will likely explore **Siamese Networks** and **Triplet Loss**, which allow a model to learn a "similarity function" rather than a standard classification. This allows the system to recognize a person it has only seen once before.

---

## One Shot Learning

This section addresses the fundamental challenge of **One-Shot Learning** in face recognition and introduces the shift from traditional classification to **Similarity Learning**.

### Key Concepts in One-Shot Learning

* **The Definition:** One-Shot Learning is the requirement to recognize a person after seeing only **one** example of their face.
* **The Failure of Traditional CNNs:**
    * Standard Softmax classification works poorly with tiny datasets (not enough data to be robust).
    * **The Scalability Problem:** If a new person joins the organization, we would have to change the output layer and retrain the entire network to include the new class.

### The Solution: Learning a Similarity Function

Rather than classifying a face directly, the goal is to learn a "Similarity Function", denoted as $d(img1, img2)$. The function inputs two images and outputs a numerical "degree of difference" (distance). A hyperparameter $\tau$ is used to make the final decision.
* If $d(img1, img2) \le \tau$, they are the same.
* If $d(img1, img2) > \tau$, they are different.

### Application to Recognition Tasks

To identify a person using a similarity function, the system performs a series of pairwise comparisons:

1. Capture the new image of the person at the door.
2. Compare that image against every person in the database using the function $d$.
3. If any comparison yields a result below the threshold $\tau$, the person is recognized.
4. If all comparisons are above $\tau$, the person is identified as "not in database."

### Benefits of the Similarity Approach

| Feature | Softmax Classification | Similarity Function ($d$) |
| --- | --- | --- |
| **New Users** | Requires retraining the model. | Just add a new photo to the database. |
| **Data Needs** | Needs many photos per person. | Works with a single photo per person. |
| **Output Type** | Class label (e.g., "User 4"). | Distance/Difference score. |

---

## Siamese Network

This section introduces the **Siamese Network**, the standard architectural solution for Face Verification. It explains how to move from "classifying" an image to "encoding" an image into a mathematical space.

### The Siamese Network Architecture

* **From Classification to Encoding:** Instead of ending with a Softmax layer to predict a name, the network ends with a **fully connected layer** that produces a feature vector (e.g., 128 numbers).
* **The Encoding ($f(x)$):** This vector is a numerical representation of the face. If the network is well-trained, similar faces will produce similar vectors.
* **Identical Twins:** A "Siamese" architecture involves running two different images ($x^{(i)}$ and $x^{(j)}$) through the **exact same network** with the **exact same weights**.
* **The Distance Function ($d$):** The difference between two people is calculated as the L2 norm (Euclidean distance) of their encodings:

$$d(x^{(i)}, x^{(j)}) = \|f(x^{(i)}) - f(x^{(j)})\|_2^2$$

### The Training Objective

The goal of backpropagation in a Siamese network is not to reduce classification error, but to optimize the parameters such that:

1. **Same Person:** If $x^{(i)}$ and $x^{(j)}$ are the same person, the distance $\|f(x^{(i)}) - f(x^{(j)})\|_2^2$ should be **small**.
2. **Different People:** If $x^{(i)}$ and $x^{(j)}$ are different people, the distance $\|f(x^{(i)}) - f(x^{(j)})\|_2^2$ should be **large**.

### Technical Insights: Why 128 Numbers?

* **Dimensionality Reduction:** We are compressing a high-resolution image (thousands of pixels) into a small "embedding" of 128 values.
* **DeepFace Influence:** This approach was popularized by the **DeepFace** [research paper](https://ieeexplore.ieee.org/document/6909616).
* **Feature Learning:** The network learns to ignore lighting, pose, and hair color, and instead focuses on the fundamental geometry of the face to create the encoding.

### Summary of Contrast Btween Traditional vs. Siamese CNN

| Feature | Traditional CNN | Siamese Network |
| --- | --- | --- |
| **Final Layer** | Softmax (Class Probabilities) | Fully Connected (Embedding Vector) |
| **Output** | A label (e.g., "ID #402") | A list of 128 numbers |
| **Comparison** | Hard to compare new classes | Easy to compare any two images |
| **Weights** | One set of weights | Two identical sets of weights |

<br>

>**Expert Note:** To actually train this, we can't use standard Cross-Entropy loss. We need a way to compare *three* images at once: a reference, a matching face, and a non-matching face. In next section, we will introduce "Triplet Loss" as the objective function.

---

## Triplet Loss

This section explains the **Triplet Loss** function, which is the primary mathematical tool used to train **Siamese Networks** for face recognition. It moves beyond simple "matching" to creating a high-dimensional space where faces are geometrically organized.

### The Core Concept: Triplets (A, P, N)

To train the network, you don't just look at one image; you look at a "triplet" of images simultaneously:

* **Anchor (A):** A reference image of a specific person.
* **Positive (P):** A different image of the same person as the anchor.
* **Negative (N):** An image of a different person.

### The Triplet Loss Formula

The goal is to ensure the distance between the Anchor and Positive is significantly smaller than the distance between the Anchor and Negative.

$$d(A, P) + \alpha \le d(A, N)$$

#### The Loss Function ($L$):

$$L(A, P, N) = \max(\|f(A) - f(P)\|^2 - \|f(A) - f(N)\|^2 + \alpha, 0)$$

* **The Margin ($\alpha$):** A hyperparameter that prevents the network from learning "trivial solutions" (like setting all encodings to zero). It forces the network to push $d(A, N)$ to be at least $\alpha$ distance further away than $d(A, P)$.
* **The Max Function:** If the model already satisfies the margin ($L < 0$), the loss is $0$. If not, the model receives a positive penalty to correct the weights.

### Selecting "Hard" Triplets

We cannot simply choose images at random. If we pick random people, the negative ($N$) will likely look so different from the anchor ($A$) that the model learns nothing (the loss is already 0).

* **"Easy" Triplets:** $d(A, P) + \alpha \ll d(A, N)$. The model learns nothing.
* **"Hard" Triplets:** We must find images where $d(A, P)$ is very close to $d(A, N)$.
* **Benefit:** Choosing hard triplets makes the gradient descent much more efficient and forces the model to learn the subtle nuances of facial features.

### Training Requirements and Scale

| Requirement | Description |
| --- | --- |
| **Training Data** | We need multiple images per person to form (A, P) pairs. |
| **Inference (One-Shot)** | Once trained, we only need one image to recognize a new person. |
| **Data Volume** | Professional systems (FaceNet, DeepFace) use 10M to 100M+ images. |
| **Strategy** | Due to the massive data needed, it is often better to use a pre-trained model parameters than training from scratch. |

### Key Takeaway

The Triplet Loss doesn't teach the model "Who is \<put your name here\>?" :D Instead, it teaches the model **"What makes two faces look like the same person?"** This creates a universal encoding that works even for people the model has never seen before.

---

## Bonus Material: Cosine Similarity vs Euclidean Distance

To implement the distance calculation between facial encodings (the 128-dimensional vectors produced by your Siamese Network), we typically use one of two metrics.

We should know that while **Euclidean Distance** is used in the Triplet Loss formula, **Cosine Similarity** is often used during "inference" (recognition) because it is more robust to changes in image lighting or contrast.

### 1. Euclidean Distance ($L2$ Norm)

This measures the straight-line distance between two points in the 128D space.


$$d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$$

### 2. Cosine Similarity

This measures the **angle** between two vectors. It tells us if the vectors are pointing in the same direction, regardless of their magnitude (length).


$$\text{Similarity} = \frac{A \cdot B}{\|A\| \|B\|}$$

### Python Implementation (using NumPy)

Here is how we can calculate both. Assume `encoding1` and `encoding2` are the $(128,)$ vectors from our model.

```python
import numpy as np

def calculate_metrics(encoding1, encoding2):
    # Euclidean Distance
    # Smaller value = More similar
    euclidean_dist = np.linalg.norm(encoding1 - encoding2)

    # Cosine Similarity
    # Value near 1.0 = More similar; Value near 0.0 = Orthogonal/Different
    dot_product = np.dot(encoding1, encoding2)
    norm_a = np.linalg.norm(encoding1)
    norm_b = np.linalg.norm(encoding2)
    cosine_sim = dot_product / (norm_a * norm_b)

    return euclidean_dist, cosine_sim

# Example usage with dummy data
enc_a = np.random.rand(128)
enc_b = np.random.rand(128)

dist, sim = calculate_metrics(enc_a, enc_b)
print(f"L2 Distance: {dist:.4f}")
print(f"Cosine Similarity: {sim:.4f}")

```

---

## Face Verification and Binary Classification

This section introduces a powerful alternative to Triplet Loss: posing face verification as a **Binary Classification** problem. Instead of comparing three images $(A, P, N)$, this method focuses on a single pair of images and learns to classify them as "Same" ($1$) or "Different" ($0$).

### Key Concepts

* **The Siamese Setup:** Like the previous method, two images are passed through identical neural networks with shared weights to produce two embeddings (feature vectors).
* **The Similarity Layer:** Instead of using a simple distance formula, the two embeddings are combined into a set of "difference features."
    * One common method is the **Element-wise Absolute Difference**: $|f(x_i)_k - f(x_j)_k|$.
    * This results in a new vector (e.g., 128 dimensions) where each value represents the disparity between specific facial features.
* **Logistic Regression Head:** These difference features are fed into a standard logistic regression unit (a dense layer with a Sigmoid activation).

$$\hat{y} = \sigma\left(\sum_{k=1}^{128} w_k \cdot |f(x^{(i)})_k - f(x^{(j)})_k| + b\right)$$

The model learns weights ($w_k$) for each feature difference, effectively learning which parts of the encoding are most important for distinguishing identities.

### Alternative Similarity Metrics

There are other mathematical ways to compare the vectors before the final classification. For example, "Chi-Square Similarity",$(\frac{(f_i - f_j)^2}{f_i + f_j})$, often used in the "DeepFace" research paper to capture non-linear relationships between the encodings.

### The Computational Trick: Pre-computation

For real-world deployment, we do not need to process every image every time:

* **Database Encodings:** We pre-calculate and store the 128-dimensional encodings for all authorized employees in a database.
* **Live Encoding:** When an employee walks up, the GPU only processes the **one** new image from the camera.
* **The Comparison:** The system only has to perform simple vector arithmetic (the subtraction and sigmoid) against the stored encodings, which is extremely fast and low-power.

### Comparison: Triplet Loss vs. Binary Classification

| Feature | Triplet Loss | Binary Classification |
| --- | --- | --- |
| **Data Unit** | Triplets (Anchor, Positive, Negative) | Pairs (Image 1, Image 2) |
| **Output** | Relative Distance | Probability ($0$ to $1$) |
| **Labeling** | Implicit (based on triples) | Explicit ($1$ for match, $0$ for non-match) |
| **Ease of Training** | Hard (requires "Hard Triplet" mining) | Easier (standard Binary Cross-Entropy) |

### Key Takeaway

Both Triplet Loss and Binary Classification rely on the **Siamese Architecture** to create meaningful encodings. While Triplet Loss is mathematically elegant for creating "clusters" of faces, the Binary Classification approach is often more intuitive to implement using standard supervised learning techniques.

---

## Bonus Material: Implementation Procedure of Binary Classification and Face Verification

With this algorithm, we are effectively talking about two different stages of the model's life: the **Training Stage** and the **Inference/Deployment Stage** (where it actually works at the door).

### 1. The Training Stage

To train the Siamese weights and the Sigmoid head, we need a **massive, labeled dataset** of pairs. We cannot train this on just two images:
* **Size of the training set:** Usually thousands or millions of pairs (e.g., the Labeled Faces in the Wild (LFW) dataset).
* **Composition of the training set:**
    * **Positive Pairs:** Two different photos of the same person ($y=1$).
    * **Negative Pairs:** Two photos of different people ($y=0$).

During this phase, the model is constantly adjusting its convolutional filters to learn what "eyes" and "noses" look like so it can create that 128D encoding. Everything is flowing through the network to calculate gradients.

### 2. The Deployment Stage

Once the model is trained and its weights are "frozen" (they are now excellent at creating face encodings), we move it to the office building. This is where pre-computation happens.

Imagine we have 100 employees.

1. **Step 1 (One-time):** We take one photo of each employee.
2. **Step 2 (The Pre-compute):** We run those 100 photos through the **trained Siamese Net** once $\rightarrow$ we get 100 different 128D vectors (encodings).
3. **Step 3 (Storage):** We throw away the images (to save VRAM/Disk space) and just save the 100 vectors in a simple database or CSV file.

### 3. The Recognition Moment: The "Test"

Now, a person walks up to the door.

1. The camera captures **one** live image ($X_{live}$).
2. The system runs $X_{live}$ through the Siamese Net to get $f(X_{live})$.
3. Instead of running the Siamese Net 100 more times for our database, we just take $f(X_{live})$ and mathematically compare it to the pre-computed vectors we stored in Step 2.3.

---

## What is Neural Style Transfer?

This section introduces **Neural Style Transfer (NST)**, one of the most visually impressive applications of Convolutional Neural Networks. It outlines the basic components of the algorithm and sets the stage for understanding how deep networks "perceive" art and objects.

### What is Neural Style Transfer?

Neural Style Transfer is an optimization technique used to take two images—a **content** image and a **style** image—and blend them together so the output image looks like the content image but is "painted" in the style of the style image.

* **Content Image ($C$):** The subject of the photo (e.g., a landscape or a landmark like the Golden Gate Bridge).
* **Style Image ($S$):** The artistic reference (e.g., Van Gogh’s *Starry Night* or a Picasso painting).
* **Generated Image ($G$):** The final result that maintains the structures of $C$ but the textures and colors of $S$.

<div style='display: flex; justify-content: center'>
    <img src='images/neural_style_transfer.png' width=750px>
</div>

### Key Concepts for Implementation

* **Layer Features:** To make NST work, we must look at the features extracted by a ConvNet at different depths:
    * **Shallow Layers:** Tend to detect simple shapes, edges, and colors.
    * **Deeper Layers:** Detect complex patterns, specific objects, and global structures.
* **The Goal:** The algorithm must find an image $G$ that minimizes a loss function involving both the content and the style.
* **Intuition Building:** Before implementing, it is necessary to visualize what individual filters in a CNN are "looking for" to understand why specific layers are better suited for "style" versus "content."

>**Expert Developer Note:** With NST we are not training weights. Instead, we are using a pre-trained network (like VGG-19) with **frozen weights** and performing gradient descent on the **pixels of the generated image ($G$) itself**. We start with a noisy image and "nudge" every pixel until it satisfies the style and content requirements.

---

## What are deep ConvNets learning?

This section explains the visual hierarchy of CNNs, based on the landmark research by [Zeiler and Fergus](https://arxiv.org/abs/1311.2901). It demonstrates how a network evolves from "seeing" simple pixels to "understanding" complex objects as data moves through deeper layers.

### Visualizing Layer Activations

* **The Visualization Method:** To understand a specific hidden unit (neuron), researchers scan the training set to find the **nine image patches** that cause that specific unit to have the highest activation.
* **Receptive Fields:** 
    * **Shallow layers** only "see" a small portion of the image (small receptive field), so their visualizations are small, simple patches.
    * **Deep layers** have a much larger receptive field; theoretically, every pixel in the input can influence a neuron in the final layers.

### The Hierarchical Evolution of Features

Here, we break down what neurons are "looking for" at each stage of the network:

<div style='display:flex; justify-content:center'>
    <img src='images/visualizing_deep_layers.png' width=750px>
</div>

| Layer | Feature Complexity | Examples of Detected Patterns |
| --- | --- | --- |
| **Layer 1** | **Simple Edges & Colors** | Lines at specific angles (diagonal, vertical), solid colors (green, orange). |
| **Layer 2** | **Textures & Basic Shapes** | Vertical stripes, round contours, thin parallel lines, honeycomb-like textures. |
| **Layer 3** | **Complex Textures & Parts** | Honeycombs, square grids, irregular patterns, and the beginnings of people or car parts. |
| **Layer 4** | **Complex Objects (Specific)** | Specific dog breeds, water, bird legs, or distinct circular shapes. |
| **Layer 5** | **Sophisticated/Varied Objects** | Varied dog species, keyboards, text-like textures, and flowers. |

### Strategic Takeaway for Neural Style Transfer

This visualization is the "secret sauce" for Neural Style Transfer:

* **Content Extraction:** If we want to keep the "content" of an image (like a dog or a building), we should look at the **deeper layers** because they capture the overall object structure.
* **Style Extraction:** If we want to capture the "style" (the textures, colors, and brushstrokes), we look at a combination of **shallower and middle layers**, which are sensitive to local textures and line orientations.

### Summary of Intuition

It's helpful to view a CNN not as a black box, but as a multi-stage filter system.

1. The first layers are our **"Eyes"** (noticing lines).
2. The middle layers are our **"Brain's Visual Cortex"** (noticing textures).
3. The final layers are our **"Cognition"** (recognizing a keyboard or a flower).

---

## Bonus: Extraction and Visualization of Features Maps of CNN

In this section, we'll implement a Python code to visualize activations at different layers of a CNN architecture. To do this, we'll use a pre-trained **VGG-19** model. VGG-19 is the industry standard for this task because its architecture is simple, linear, and very effective at capturing the hierarchical features.

### Extracting Layer Activations

First, we define a model that doesn't output classifications, but rather the "internal thoughts" (activations) of the specific layers we are interested in. For demonstration purpose, we will look at activations of three convolutional layers at different blocks (i.e., `block1_conv1`, `block3_conv1`, `block5_conv1`):

<div style='display: flex; justify-content: center'>
    <img src='images/vgg19.png' width=750px>
</div>

<br>

| Layer | Dimensions | No. of Filters | Feature Type |
| --- | --- | --- | --- |
| **Input** | $(224, 224, 3)$ | $3$ (RGB) | Raw Pixels |
| **Block 1** | $(224, 224, 64)$ | $64$ | Low-level (Edges) |
| **Block 3** | $(56, 56, 256)$ | $256$ | Mid-level (Textures) |
| **Block 5** | $(14, 14, 512)$ | $512$ | High-level (Objects) |


Here is Python code for extraction of feature maps at three different layers:


```python
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

# Load VGG-19 pre-trained on ImageNet
vgg = tf.keras.applications.VGG19(include_top=False, weights='imagenet')
vgg.trainable = False 

# Select layers to visualize (from shallow to deep)
# Block1_conv1: Edges/Colors
# Block3_conv1: Textures
# Block5_conv1: Complex Objects
layer_names = ['block1_conv1', 'block3_conv1', 'block5_conv1']
outputs = [vgg.get_layer(name).output for name in layer_names]

# Create a "Feature Extractor" model
# This model takes an image and outputs the intermediate activations
activation_model = tf.keras.Model(inputs=vgg.input, outputs=outputs)

```

### Visualizing the Features

Once we have the activations, they are essentially 3D tensors: $(Height, Width, Channels)$. To visualize them, we plot individual "channels" (filters). Each channel represents one specific feature the network is looking for.

```python
def visualize_layers(img_path):
    # Load and preprocess image
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
    img_tensor = tf.keras.preprocessing.image.img_to_array(img)
    img_tensor = np.expand_dims(img_tensor, axis=0)
    img_tensor = tf.keras.applications.vgg19.preprocess_input(img_tensor)

    # Get activations
    activations = activation_model.predict(img_tensor)

    # Plotting
    fig, axes = plt.subplots(1, len(layer_names), figsize=(15, 5))
    
    for i, activation in enumerate(activations):
        # We pick the first filter (channel 0) of each layer to display
        # In a real scenario, you'd loop through many filters
        feature_map = activation[0, :, :, 0] 
        
        axes[i].imshow(feature_map, cmap='viridis')
        axes[i].set_title(f"Layer: {layer_names[i]}")
        axes[i].axis('off')
    
    plt.show()

# Run visualization
visualize_layers('your_image.jpg')

```

### Understanding the Outputs

When we run the above code, we will see a progression of abstraction:

1. **`block1_conv1` (Shallow):** The image will look like a highlighted version of the original. we’ll see glowing lines where the edges of objects are.
2. **`block3_conv1` (Middle):** The original image becomes harder to recognize. The map will highlight repeating patterns or specific textures.
3. **`block5_conv1` (Deep):** The output is very sparse and "bloppy." Only very specific regions will light up—these are the neurons that have identified "this is a cat's ear", for example.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

# Load VGG-19 pre-trained on ImageNet
vgg = tf.keras.applications.VGG19(include_top=False, weights='imagenet')
vgg.trainable = False 

# Select layers to visualize (from shallow to deep)
# Block1_conv1: Edges/Colors
# Block3_conv1: Textures
# Block5_conv1: Complex Objects
layer_names = ['block1_conv1', 'block3_conv1', 'block5_conv1']
outputs = [vgg.get_layer(name).output for name in layer_names]

# Create a "Feature Extractor" model
# This model takes an image and outputs the intermediate activations
activation_model = tf.keras.Model(inputs=vgg.input, outputs=outputs)


def visualize_layers(img_path):
    # Load and preprocess image
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
    img_tensor = tf.keras.preprocessing.image.img_to_array(img)
    img_tensor = np.expand_dims(img_tensor, axis=0)
    img_tensor = tf.keras.applications.vgg19.preprocess_input(img_tensor)

    # Get activations (forward pass)
    activations = activation_model.predict(img_tensor)

    print(activations[1].shape)
    # Plotting
    fig, axes = plt.subplots(1, len(layer_names), figsize=(15, 5))
    
    for i, activation in enumerate(activations):
        # We pick the first filter (channel 0) of each layer to display
        # In a real scenario, you'd loop through many filters
        feature_map = activation[0, :, :, 0] 
        
        axes[i].imshow(feature_map, cmap='viridis')
        axes[i].set_title(f"Layer: {layer_names[i]}")
        axes[i].axis('off')
    
    plt.show()

# Run visualization
visualize_layers('images/cat.jpg')

---

## Cost Function

This section defines the **Optimization Framework** for Neural Style Transfer (NST). We'll find this fascinating because it flips the usual machine learning paradigm on its head: instead of optimizing **weights** ($W$), we are optimizing the **input pixels** ($G$) themselves.

* **Problem Formulation:** We start with a Content image ($C$) and a Style image ($S$). The goal is to produce a Generated image ($G$).
* **The Total Cost Function ($J$):** The quality of $G$ is determined by a combined mathematical score:

    $$J(G) = \alpha J_{content}(C, G) + \beta J_{style}(S, G)$$

    * **$J_{content}(C, G)$:** Measures how much the shapes and objects in $G$ match those in $C$.
    * **$J_{style}(S, G)$:** Measures how much the textures and colors in $G$ match those in $S$.

* **Hyperparameters ($\alpha, \beta$):** These control the "artistic balance."
    * A high **$\alpha$** keeps the result very realistic (more like the photo).
    * A high **$\beta$** makes the result very abstract (more like the painting).

### The Algorithm Workflow

Unlike standard training, the "model" (a pre-trained ConvNet like VGG-19) remains frozen. The pixels of $G$ are the only variables that change.

1. **Initialization:** Create $G$ as a "White Noise" image (pixels are just random numbers).
2. **Forward Pass:** Pass $C, S,$ and $G$ through the pre-trained network to extract feature maps.
3. **Compute Loss:** Calculate $J(G)$ by comparing the feature maps of $G$ against $C$ and $S$.
4. **Gradient Descent:** Update the pixels of $G$ to reduce the loss:

$$G := G - \eta \frac{\partial J(G)}{\partial G}$$

5. **Iteration:** Repeat until the white noise transforms into a recognizable piece of art.

<div style='display: flex; justify-content: center'>
    <img src='images/nst_c_s.png' width=400>
</div>

<div style='display: flex; justify-content: center'>
    <img src='images/nst_g.png' width=200>
</div>


### Why the Random Start?

Many modern developers initialize $G$ with the **Content Image $C$**. This significantly speeds up convergence and often results in a cleaner final image because the model doesn't have to "learn" the shapes from scratch—it just has to "re-paint" them.